In [1]:
!pip install -q sentence-transformers rapidfuzz evaluate bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00


In [16]:
from utils import (
    download_dataset, load_parsed, sample_conversations,
    build_context, build_global_catalog,
    build_train_freq, build_recommender_references,
    evaluate_soft, evaluate_novelty, evaluate_bertscore, run_full_evaluation, _norm_title
)

In [12]:
def audit(outputs: list, parsed: list, k: int = 5) -> dict:
    """Audita comportamientos problemáticos en las predicciones: eco (parroting),
    recomendaciones imposibles de acertar (post-fecha del dataset) y alucinación
    de formato (sin año).

    Usa build_context(d) como referencia de "lo visible" -- coherente con el
    protocolo actual (contexto completo sin truncar, oculto solo el último
    turno del recommender).

    Args:
        outputs: Lista de dicts {"idx", "predicted", ...}.
        parsed: Dataset parseado completo.
        k: Cuántas predicciones por diálogo considerar.

    Returns:
        Dict con echo_rate, post2018_rate, noyear_rate, n_pred.
    """
    yr = re.compile(r'\((\d{4})\)')
    n_pred = echo = post18 = noyear = 0

    for o in outputs:
        d = parsed[o["idx"]]
        ctx = build_context(d).lower()
        for p in o["predicted"][:k]:
            n_pred += 1
            t = _norm_title(p)
            if t and t in ctx:
                echo += 1
            m = yr.search(p)
            if m:
                if int(m.group(1)) > 2018:
                    post18 += 1
            else:
                noyear += 1

    if not n_pred:
        print("Audit: sin predicciones.")
        return {"echo_rate": 0.0, "post2018_rate": 0.0, "noyear_rate": 0.0, "n_pred": 0}

    result = {
        "echo_rate": round(echo / n_pred, 4),
        "post2018_rate": round(post18 / n_pred, 4),
        "noyear_rate": round(noyear / n_pred, 4),
        "n_pred": n_pred,
    }

    print(f"Recomendaciones auditadas: {n_pred}")
    print(f"  Eco (anti-parrot violado): {result['echo_rate']*100:.1f}%  (debe ser ~0)")
    print(f"  Post-2018 (miss seguro):   {result['post2018_rate']*100:.1f}%")
    print(f"  Sin año (proxy alucinación): {result['noyear_rate']*100:.1f}%")

    return result

In [4]:
# from utils import download_dataset, load_parsed, build_train_freq, build_recommender_references
DATASET = "pearl"

paths = download_dataset("pearl", splits=("train", "test"))
train_parsed = load_parsed(paths["train"],"pearl")

test_parsed = load_parsed(paths["test"], "pearl")

train_freq, n_train = build_train_freq(paths["train"],"pearl")
references = build_recommender_references(paths["test"], only_movie_turns=True, dataset="pearl")

Descargando pearl_train.json...
Descargando pearl_test.json...
Train freq construido: 9303 títulos únicos en 50000 diálogos
Referencias construidas: 2277 diálogos


In [5]:
paths


{'train': 'pearl_train.json', 'test': 'pearl_test.json'}

In [6]:
from sentence_transformers import SentenceTransformer
import torch
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
ace_base_results_path="/content/ace_results_base_pearl.json"
# ace_tools_results_path="/content/ace_results_tools_pearl.json"

playbook_base_path="/content/playbook_final_base_pearl.json"
# playbook_tools_path="/content/playbook_final_tools_pearl.json"

In [8]:
# cambiar nombres

import json

with open(ace_base_results_path) as f:   # o el path donde estén
    out_base = json.load(f)

with open(playbook_base_path) as f:  # opcional, solo si quieren inspeccionarlo
    playbook_base = json.load(f)

print(f"Outputs cargados: {len(out_base)} diálogos")

Outputs cargados: 300 diálogos


In [9]:
print(len(out_base), "entradas")
print(out_base[0])
print(out_base[-1])

300 entradas
{'idx': 0, 'predicted': ['The Last Boy Scout (1991)', 'Misery (1990)', 'The Sixth Sense (1999)', 'The Negotiator (1998)', '12 Monkeys (1995)'], 'applied_bullets': ['RULE-178', 'RULE-172', 'RULE-174'], 'message': "I recommend 'The Last Boy Scout' (1991) because it features Bruce Willis in a gritty, intense role with a memorable opening scene and delivers the action-packed, entertaining tone you're looking for. You might also enjoy 'Misery' (1990), 'The Sixth Sense' (1999), 'The Negotiator' (1998), and '12 Monkeys' (1995).", 'raw': '{\n  "message": "I recommend \'The Last Boy Scout\' (1991) because it features Bruce Willis in a gritty, intense role with a memorable opening scene and delivers the action-packed, entertaining tone you\'re looking for. You might also enjoy \'Misery\' (1990), \'The Sixth Sense\' (1999), \'The Negotiator\' (1998), and \'12 Monkeys\' (1995).",\n  "structured_recommendations": ["The Last Boy Scout (1991)", "Misery (1990)", "The Sixth Sense (1999)", 

In [ ]:
# with open(ace_tools_results_path) as f:   # o el path donde estén
#     out_tools = json.load(f)

# with open(playbook_tools_path) as f:  # opcional, solo si quieren inspeccionarlo
#     playbook_tools = json.load(f)

# print(f"Outputs cargados: {len(out_tools)} diálogos")

Outputs cargados: 300 diálogos


In [10]:


# adaptar a pearl, en especial lo de gt field si es que aplia
metrics_base = run_full_evaluation(
    out_base, test_parsed, EMBED_MODEL,
    train_freq, n_train, references,
    threshold=0.90, gt_field="gt_accepted"
)

  EVALUACIÓN COMPLETA

── Recall & MRR (soft-matching) ──
Evaluable: 300 / 300
  Recall@1: 0.0100
  Recall@5: 0.0333
  MRR:      0.0190

── Novelty ──
  Novelty:  10.39 bits (norm: 0.666)
  No vistas en train: 16.2% (236/1455)

── BERTScore ──


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  BERTScore (291/300): P=0.8725  R=0.8843  F1=0.8782



In [11]:
metrics_base

{'Recall@1': 0.01,
 'Recall@5': 0.0333,
 'MRR': 0.019,
 'n_evaluable': 291,
 'novelty_mean': 10.3941,
 'novelty_norm': 0.6659,
 'unseen_rate': 0.1622,
 'n_dialogos': 291,
 'n_recs': 1455,
 'precision': 0.8725,
 'recall': 0.8843,
 'f1': 0.8782,
 'n_total': 300}

In [14]:
import re

In [17]:
audit(out_base, test_parsed)

Recomendaciones auditadas: 1455
  Eco (anti-parrot violado): 0.4%  (debe ser ~0)
  Post-2018 (miss seguro):   7.2%
  Sin año (proxy alucinación): 0.1%


{'echo_rate': 0.0041,
 'post2018_rate': 0.0722,
 'noyear_rate': 0.0007,
 'n_pred': 1455}

In [ ]:
# metrics_tools = run_full_evaluation(
#     out_tools, test_parsed, EMBED_MODEL,
#     train_freq, n_train, references,
#     threshold=0.90, gt_field="gt_accepted"
# ) # son iguales porque corrió sin tools 2 veces, correr de nuevo

  EVALUACIÓN COMPLETA

── Recall & MRR (soft-matching) ──
Evaluable: 300 / 300
  Recall@1: 0.0100
  Recall@5: 0.0333
  MRR:      0.0190

── Novelty ──
  Novelty:  10.39 bits (norm: 0.666)
  No vistas en train: 16.2% (236/1455)

── BERTScore ──


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  BERTScore (291/300): P=0.8725  R=0.8843  F1=0.8782

